# InjectNo — Notebook 14: Reproduce the headline numbers (CPU, ~1 minute, no token needed)
Downloads the recorded agent transcripts and scores from the public dataset
[kenil1710/injectno-bench](https://huggingface.co/datasets/kenil1710/injectno-bench) and recomputes every number in the article's
results table, with 95% Wilson intervals. Nothing here calls a model.

In [ ]:
!pip install -q -U pandas huggingface_hub

In [ ]:
import math, json, pandas as pd
from huggingface_hub import hf_hub_download
REPO = "kenil1710/injectno-bench"; R5 = "injectno-aya-expanse-8b-v3"
get = lambda p: hf_hub_download(REPO, p, repo_type="dataset")
v3 = pd.read_csv(get(f"results/{R5}/{R5}_eval_v3.csv"))
hw = pd.read_csv(get(f"results/{R5}/{R5}_eval_handwritten.csv"))
v2 = pd.read_csv(get(f"results/{R5}/{R5}_eval_v2.csv"))
v1 = pd.read_csv(get(f"results/{R5}/{R5}_eval_v1.csv"))
for d in (v3, hw, v2):
    for c in ("harmful", "correct", "warned"): d[c] = d[c].astype(bool)
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = p + z*z/(2*n); r = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return max(0, (c - r) / d), min(1, (c + r) / d)
def f(s):
    k, n = int(s.sum()), len(s); lo, hi = wilson(k, n)
    return f"{100*k/n:.1f}% ({k}/{n}) [{100*lo:.1f}–{100*hi:.1f}]"
print("rows:", {"v3": len(v3), "handwritten": len(hw), "v2": len(v2), "v1": len(v1)})

## Benchmark v3 test (base vs run 4 vs InjectNo run 5)

In [ ]:
a, b = v3[v3.kind == "attack"], v3[v3.kind == "benign"]
t = pd.DataFrame({"attacks that worked": a.groupby("model").harmful.apply(f), "warned on attacks": a.groupby("model").warned.apply(f),
                  "normal emails done right": b.groupby("model").correct.apply(f)})
print(t.to_string())
print("\nheld-out attacks that worked (%):"); print(a.groupby(["held_out", "model"]).harmful.mean().mul(100).round(1).unstack().to_string())
redir = a[a.family.isin(["inv_inject", "fwd_inject", "bec_multi"])]
print("\nredirection families only (inv/fwd/bec):"); print(redir.groupby("model").harmful.apply(f).to_string())

## Hand-written emails (set 1 = author's, set 2 = dev, set 3 = final exam)

In [ ]:
print(hw.groupby("model").correct.apply(f).to_string(), "\n")
s3 = hw[hw.set == "set3_final24"]
print("final exam correct:"); print(s3.groupby("model").correct.apply(f).to_string())
print("final exam harmful:"); print(s3.groupby("model").harmful.sum().to_string())
print("\nhand-written attacks that worked:"); print(hw[hw.kind == "attack"].groupby("model").harmful.apply(f).to_string())

## Regression checks for run 5: benchmark v2 test and the original 512-email v1 test

In [ ]:
a2, b2 = v2[v2.kind == "attack"], v2[v2.kind == "benign"]
print("v2 attacks that worked:", f(a2.harmful), "| v2 normal correct:", f(b2.correct))
v1["hijacked"] = v1.hijacked.astype(bool); v1["task_ok"] = v1.task_ok.astype(bool)
print("v1 attack success:", f(v1[v1.kind == "attack"].hijacked), "| v1 normal tasks done:", f(v1[v1.kind == "benign"].task_ok))

## Lexsi AuditKIT, SafeTune and Prompt Guard results

In [ ]:
au = pd.read_csv(get("results/auditkit/injectno_auditkit_audit.csv"))
au["audit_pass"] = au.audit_verdict.eq("success")
print("AuditKIT outcome pass rate (actions only):"); print(au.groupby(["set", "kind", "model"]).audit_pass.mean().mul(100).round(1).unstack().to_string())
print("\nAuditKIT diagnostics (v3):"); print(au[au.set == "v3"].groupby("model")[["tool_call_validity", "tool_permission", "redundant_tool_calls"]].mean().round(3).to_string())
print("\nharmful-by-our-scorer but passed by AuditKIT:", int((au.harmful.astype(bool) & au.audit_pass).sum()))
st = json.load(open(get("results/safetune/injectno_safetune_drift.json")))
print("\nSafeTune:"); print(json.dumps(st, indent=1)[:1500])
pg = pd.read_csv(get("results/promptguard/injectno_promptguard_scores.csv"))
print("\nPrompt Guard: emails flagged =", int(pg.drop_duplicates(["set", "id"]).pg_flag.sum()), "of", len(pg.drop_duplicates(["set", "id"])))
try:
    print(json.dumps(json.load(open(get("results/promptguard/injectno_promptguard_sanity.json"))), ensure_ascii=False)[:800])
except Exception as e: print("sanity file:", e)

## Prompt-only defence baseline (NB13), if uploaded

In [ ]:
try:
    pdf = pd.read_csv(get("results/prompt_defence/injectno_prompt_defence_eval.csv"))
    for c in ("harmful", "correct", "warned"): pdf[c] = pdf[c].astype(bool)
    for name, d in (("v3", pdf[pdf.set == "v3"]), ("hand-written", pdf[pdf.set != "v3"])):
        a, b = d[d.kind == "attack"], d[d.kind == "benign"]
        print(name); print(pd.DataFrame({"attacks that worked": a.groupby("model").harmful.apply(f), "normal emails done right": b.groupby("model").correct.apply(f)}).to_string())
except Exception as e:
    print("prompt-defence results not found:", repr(e)[:120])